# 08 — Enhancement Round 2 (adjudication)

Two top-priority tracks from the Round-2 plans: a Title×Pclass joint feature (representation) and repeated-CV adjudication of the Round-1 challengers (validation). Both pre-committed gates; both reject. No test labels, no downloads, no leaderboard use.


In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))


In [ ]:
R = json.loads((ROOT/'reports'/'experiment_results.json').read_text())
A = json.loads((ROOT/'reports/gap3_r2a.json').read_text())
B = json.loads((ROOT/'reports/gap3_r2b.json').read_text())
RF = R['best_cv']
print('reference RF:', round(RF, 4))

## R2A: Title×Pclass — dead flat, REJECT on every gate


In [ ]:
assert abs(A['results']['base_random']['mean'] - RF) < 1e-9
assert A['verdict'] == 'REJECT'
assert all(g['passed'] is False for g in A['gates'].values()), 'a gate unexpectedly passed'
print('variant random %.6f (Delta %+.6f)' % (A['results']['variant_random']['mean'], A['results']['delta_random']))
print('variant group  %.6f (Delta %+.6f)' % (A['results']['variant_group']['mean'], A['results']['delta_group']))
print('fold wins random %d/5, group %d/5' % (A['results']['fold_wins_random']['wins'], A['results']['fold_wins_group']['wins']))
print('R2A confirmed: joint code adds nothing measurable on any gate')

## R2B: repeated adjudication — OVERTURN to plateau
Single-seed suggested C1/V3 ahead; over the same 25 repeated partitions neither clears the pre-committed bar (delta ≥ +0.0134 AND paired-t ≥ 2.0 AND mean25 > 0.8440).


In [ ]:
m = {k: B['repeated25'][k]['mean25'] for k in B['repeated25']}
print('mean25:', {k.split('_')[0]: round(v, 4) for k, v in m.items()})
for k in ('C1_minus_R0', 'V3_minus_R0'):
    p = B['paired'][k]
    print('%s: delta %+.4f t=%.2f' % (k, p['mean_diff'], p['t_stat']))
assert B['verdict'] == 'OVERTURN'
assert B['gates']['C1']['confirm_all'] is False and B['gates']['V3']['confirm_all'] is False
assert B['paired']['C1_minus_R0']['t_stat'] >= 2.0 and B['paired']['C1_minus_R0']['mean_diff'] < 0.0134
print('R2B confirmed: t-statistic without magnitude is not adoption evidence')

## Reconciling the two repeated-CV means (executed, not hand-waved)
Worker 5 reported repeated-5×5 mean 0.8352 with 25 listed fold scores; R2B recomputed 0.8366 on its own 25 partitions (five separate seed splitters instead of one RepeatedStratifiedKFold object). Both stand — different valid instantiations, Delta=0.0014:


In [ ]:
w5_folds = [0.860335,0.837079,0.820225,0.848315,0.853933,0.826816,0.831461,0.786517,0.820225,0.842697,
 0.854749,0.825843,0.870787,0.842697,0.814607,0.860335,0.780899,0.814607,0.786517,0.882022,
 0.843575,0.825843,0.853933,0.848315,0.848315]  # W5-reported repeated folds, verbatim
import numpy as np
assert abs(float(np.mean(w5_folds)) - 0.8352) < 5e-4, 'W5 aggregate inconsistent with its folds'
S = json.loads((ROOT/'reports/gap_stability.json').read_text())
assert abs(S['repeated']['mean'] - float(np.mean(w5_folds))) < 5e-4
print('W5 stored aggregate 0.8352 == mean of its listed folds: file is correct')
print('R2B 0.8366 differs only by partition instantiation (Delta=0.0014: instantiation noise, cf. seed range 0.0134)')
print('Lesson recorded: even the robustness estimator wobbles ~0.001-0.002 by construction - another reason single-seed +0.003 deltas cannot promote challengers')

## Round-2 outcome: RF retained again; plateau deepens honestly

- R2A kills the best-motivated representation idea flat (all five gates fail, including slice checks).
- R2B kills single-seed promotion logic: t ≥ 2 without magnitude is insufficient, and repeated means (~0.835–0.837) sit below seed-42 single (0.8440) for every candidate.
- Production model and submission unchanged. Recorded as E16–E17.
- Standing lesson: at n=891 with ±0.014 fold noise, only deltas clearly outside the ~0.013 seed range merit adoption. Nothing tested so far qualifies.
